In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
class Dataset:
    pass
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

FIX_CATE_DATA_PD = pd.DataFrame({"x1": [1, 2, 3], "x2": [4, 5, 6], "y": [0, 1, 0], "w": [1.0, 0.5, 1.5]})
FIX_CATE_DATA_PL = pl.from_pandas(FIX_CATE_DATA_PD)
FIX_CATE_X_COLUMNS = ["x1", "x2"]
FIX_CATE_Y_COLUMNS = ["y"]
FIX_CATE_W_COLUMNS = ["w"]

class _DatasetShell:
    pass

def _make_private_self(data, x_columns=FIX_CATE_X_COLUMNS, y_columns=FIX_CATE_Y_COLUMNS, w_columns=FIX_CATE_W_COLUMNS):
    obj = SimpleNamespace(x_columns=x_columns, y_columns=y_columns, w_columns=w_columns)
    setattr(obj, "__df", data)
    return obj

# --- dataset_save_load ---
FIX_DATASET_SAVE_LOAD_DATA = FIX_CATE_DATA_PL
FIX_DATASET_SAVE_LOAD_PATH = Path("cate_dataset_saved")
FIX_DATASET_SAVE_LOAD_SAVE = lambda path: None

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_dataset_df_clone():
    class Dataset:
        def __init__(
            self,
            df: pd.DataFrame,
            x_columns: list[str],
            y_columns: list[str],
            w_columns: list[str],
        ) -> None:
            self.x_columns = x_columns
            self.y_columns = y_columns
            self.w_columns = w_columns
            self.__df = df.copy()
            self._validate(
                self.__df.columns.to_list(), self.x_columns, self.y_columns, self.w_columns
            )

        @property
        def X(self) -> pd.DataFrame:
            return self.__df.loc[:, self.x_columns].copy()

        @property
        def y(self) -> pd.DataFrame:
            return self.__df.loc[:, self.y_columns].copy()

        @property
        def w(self) -> pd.DataFrame:
            return self.__df.loc[:, self.w_columns].copy()

        def to_pandas(self) -> pd.DataFrame:
            return self.__df.copy()
    return Dataset

def before_dataset_loc_select():
    def X(self) -> pd.DataFrame:
        return self.__df.loc[:, self.x_columns].copy()
    def y(self) -> pd.DataFrame:
        return self.__df.loc[:, self.y_columns].copy()
    def w(self) -> pd.DataFrame:
        return self.__df.loc[:, self.w_columns].copy()
    return X, y, w

def before_dataset_save_load(data, path, save):
    def save(self, path: Path) -> None:
        path.mkdir(exist_ok=True, parents=True)
        self.__df.to_csv(path / "data.csv", index=False)
        json.dump(
            {
                "x_columns": self.x_columns,
                "y_columns": self.y_columns,
                "w_columns": self.w_columns,
            },
            (path / "meta.json").open("w"),
        )

    @classmethod
    def load(cls, path: Path) -> Dataset:
        data_path = path / "data.csv"
        meta_path = path / "meta.json"
        if (not data_path.exists()) or (not meta_path.exists()):
            raise FileNotFoundError()

        df = pd.read_csv(data_path)
        meta = json.load(meta_path.open(mode="r"))
        return cls(df, **meta)
    return load

def before_dataset_to_pandas():
    def to_pandas(self) -> pd.DataFrame:
        return self.__df.copy()
    return to_pandas


In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_dataset_df_clone():


    class Dataset:
        def __init__(
            self,
            df: pl.DataFrame,
            x_columns: list[str],
            y_columns: list[str],
            w_columns: list[str],
        ) -> None:
            self.x_columns = x_columns
            self.y_columns = y_columns
            self.w_columns = w_columns
            self.__df = df.clone()
            self._validate(self.__df.columns, self.x_columns, self.y_columns, self.w_columns)

        @property
        def X(self) -> pl.DataFrame:
            return self.__df.select(self.x_columns).clone()

        @property
        def y(self) -> pl.DataFrame:
            return self.__df.select(self.y_columns).clone()

        @property
        def w(self) -> pl.DataFrame:
            return self.__df.select(self.w_columns).clone()

        def to_pandas(self) -> pl.DataFrame:
            return self.__df.clone()
    return Dataset

def gen_dataset_loc_select():


    def X(self) -> pl.DataFrame:
        return self.__df.select(self.x_columns).clone()


    def y(self) -> pl.DataFrame:
        return self.__df.select(self.y_columns).clone()


    def w(self) -> pl.DataFrame:
        return self.__df.select(self.w_columns).clone()
    return X, y, w

def gen_dataset_save_load():
    import json
    from pathlib import Path

    class Dataset:
        def __init__(self, df, x_columns, y_columns, w_columns):
            self.__df = df
            self.x_columns = x_columns
            self.y_columns = y_columns
            self.w_columns = w_columns

        def save(self, path: Path) -> None:
            path.mkdir(exist_ok=True, parents=True)
            self.__df.write_csv(path / "data.csv")
            json.dump(
                {
                    "x_columns": self.x_columns,
                    "y_columns": self.y_columns,
                    "w_columns": self.w_columns,
                },
                (path / "meta.json").open("w"),
            )


        @classmethod
        def load(cls, path: Path) -> "Dataset":
            data_path = path / "data.csv"
            meta_path = path / "meta.json"
            if (not data_path.exists()) or (not meta_path.exists()):
                raise FileNotFoundError()

            df = pl.read_csv(data_path)
            meta = json.load(meta_path.open(mode="r"))
            return cls(df, **meta)
    return Dataset

def gen_dataset_to_pandas():

    def to_pandas(self) -> pl.DataFrame:
        return self.__df.clone()
    return to_pandas

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:

# === Tests: dataset_loc_select ===

try:
    _gen_functions = gen_dataset_loc_select()
    _r = [_fn(_make_private_self(FIX_CATE_DATA_PL)) for _fn in _gen_functions]
    print("✅ L1 smoke gen_dataset_loc_select: OK, outputs=", len(_r))
except Exception as _e:
    print(f"❌ L1 smoke gen_dataset_loc_select: {type(_e).__name__}: {_e}")

try:
    _before_functions = before_dataset_loc_select()
    _rb = [_fn(_make_private_self(FIX_CATE_DATA_PD)) for _fn in _before_functions]
    print("✅ L1 smoke before_dataset_loc_select: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_dataset_loc_select: {type(_e).__name__}: {_e}")

try:
    _before_functions = before_dataset_loc_select()
    _gen_functions = gen_dataset_loc_select()
    for _name, _before_fn, _gen_fn in zip(["X", "y", "w"], _before_functions, _gen_functions):
        compare(
            _before_fn(_make_private_self(FIX_CATE_DATA_PD)),
            _gen_fn(_make_private_self(FIX_CATE_DATA_PL)),
            f"dataset_loc_select {_name}",
            check_row_order=True,
        )
except Exception as _e:
    print(f"❌ L2 equivalence dataset_loc_select: setup error — {type(_e).__name__}: {_e}")

try:
    _before_functions = before_dataset_loc_select()
    _gen_functions = gen_dataset_loc_select()
    for _name, _before_fn, _gen_fn in zip(["X", "y", "w"], _before_functions, _gen_functions):
        compare(
            _before_fn(_make_private_self(FIX_CATE_DATA_PD.head(0))),
            _gen_fn(_make_private_self(FIX_CATE_DATA_PL.head(0))),
            f"L3 edge dataset_loc_select empty {_name}",
            check_row_order=True,
        )
except Exception as _e:
    print(f"❌ L3 edge dataset_loc_select empty: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_dataset_loc_select: OK, outputs= 3
✅ L1 smoke before_dataset_loc_select: OK
✅ L2 equivalence dataset_loc_select X: MATCH
✅ L2 equivalence dataset_loc_select y: MATCH
✅ L2 equivalence dataset_loc_select w: MATCH
✅ L3 edge dataset_loc_select empty X: MATCH
✅ L3 edge dataset_loc_select empty y: MATCH
✅ L3 edge dataset_loc_select empty w: MATCH
